# 第15章 機械学習アプリの作り方

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍のリスト番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/appendix/

## モデルの保存と再利用

### pickleによるモデルの保存と読み込み

**リスト 15.1**　`pickle`によるモデルの保存

In [ ]:
import pickle
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# モデルの学習（iris は seaborn 版で読み込み）
df_iris = sns.load_dataset("iris")
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train, X_test, y_train, y_test = train_test_split(
    df_iris[feature_cols].values, df_iris["species"],
    test_size=0.2, random_state=42
)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"学習時の精度: {model.score(X_test, y_test):.4f}")

# モデルをファイルに保存
with open("iris_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("モデルを iris_model.pkl に保存しました")

**リスト 15.2**　保存したモデルの読み込みと予測

In [ ]:
import pickle
import numpy as np

# モデルの読み込み
with open("iris_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# 読み込んだモデルで予測
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
prediction = loaded_model.predict(sample)
proba = loaded_model.predict_proba(sample)

print(f"予測結果: {prediction[0]}")
print(f"確率: {dict(zip(loaded_model.classes_, proba[0].round(4)))}")

### joblibによる保存

**リスト 15.3**　`joblib`によるモデルの保存と読み込み

In [ ]:
import joblib

# モデルの保存
joblib.dump(model, "iris_model.joblib")
print("モデルを iris_model.joblib に保存しました")

# モデルの読み込み
loaded_model = joblib.load("iris_model.joblib")
print(f"読み込んだモデルの精度: {loaded_model.score(X_test, y_test):.4f}")

## 推論処理の作り方

### 予測用モデルの準備

**リスト 15.4**　ペンギン分類モデルの学習と保存

In [ ]:
# train_penguin.py
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# データの準備（推論時は NumPy 配列を渡すため、.values で学習しておく）
penguins = sns.load_dataset("penguins").dropna()
X = penguins[["bill_length_mm", "bill_depth_mm",
              "flipper_length_mm", "body_mass_g"]].values
y = penguins["species"]

# パイプラインの構築と学習
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=42
    ))
])
pipeline.fit(X, y)

# モデルの保存
joblib.dump(pipeline, "penguin_model.joblib")
print("モデルを保存しました")
print(f"クラス: {list(pipeline.classes_)}")

### 推論関数の作成

**リスト 15.5**　推論関数`predict_penguin`の作成

In [ ]:
# predict.py
import joblib
import numpy as np

# モデルの読み込み（アプリ起動時に1回だけ実行する想定）
model = joblib.load("penguin_model.joblib")

def predict_penguin(bill_length, bill_depth,
                    flipper_length, body_mass):
    """特徴量からペンギンの種類と各クラスの確率を返す"""
    features = np.array([[bill_length, bill_depth,
                          flipper_length, body_mass]])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    proba_dict = {
        species: round(float(prob), 4)
        for species, prob in zip(model.classes_, probabilities)
    }
    return prediction, proba_dict

# 動作確認
species, proba = predict_penguin(39.1, 18.7, 181.0, 3750.0)
print(f"予測結果: {species}")
print(f"確率: {proba}")

## UIの作り方

### 最初のStreamlitアプリ

**リスト 15.6**　最初のStreamlitアプリ

In [ ]:
# app.py
import streamlit as st

st.title("はじめての Streamlit アプリ")
st.write("Streamlit を使えば、Python だけで Web アプリが作れます！")

# テキスト入力
name = st.text_input("名前を入力してください")
if name:
    st.write(f"こんにちは、{name}さん！")

# スライダー
age = st.slider("年齢を選択してください", 0, 100, 20)
st.write(f"あなたの年齢: {age}歳")

### ペンギン分類アプリの作成

**リスト 15.7**　ペンギン分類アプリの作成

In [ ]:
# penguin_app.py
import streamlit as st
import joblib
import numpy as np

# モデルの読み込み（キャッシュで高速化）
@st.cache_resource
def load_model():
    return joblib.load("penguin_model.joblib")

model = load_model()

# アプリのタイトル
st.title("ペンギン種類分類アプリ")
st.write("ペンギンの体の特徴を入力すると、種類を予測します。")

# サイドバーに入力フォームを配置
st.sidebar.header("特徴量の入力")
bill_length = st.sidebar.slider(
    "くちばしの長さ (mm)", 30.0, 60.0, 45.0)
bill_depth = st.sidebar.slider(
    "くちばしの奥行き (mm)", 13.0, 22.0, 17.0)
flipper_length = st.sidebar.slider(
    "翼の長さ (mm)", 170.0, 235.0, 200.0)
body_mass = st.sidebar.slider(
    "体重 (g)", 2700.0, 6300.0, 4200.0)

# 予測の実行
features = np.array(
    [[bill_length, bill_depth, flipper_length, body_mass]]
)
prediction = model.predict(features)[0]
probabilities = model.predict_proba(features)[0]

# 結果の表示
st.subheader("予測結果")
st.write(f"**予測された種類: {prediction}**")

# 確率の表示
st.subheader("各種類の確率")
for species, prob in zip(model.classes_, probabilities):
    st.write(f"{species}: {prob:.1%}")
    st.progress(float(prob))

### 感情分析アプリの作成

**リスト 15.8**　感情分析アプリの作成

In [ ]:
# sentiment_app.py
import streamlit as st
from transformers import pipeline

# モデルの読み込み（キャッシュで高速化）
@st.cache_resource
def load_classifier():
    return pipeline(
        "sentiment-analysis",
        model="lxyuan/distilbert-base-multilingual"
              "-cased-sentiments-student"
    )

classifier = load_classifier()

# アプリのタイトル
st.title("感情分析アプリ")
st.write("テキストを入力すると、感情を判定します。")

# テキスト入力
text = st.text_area(
    "分析したいテキストを入力してください",
    height=100,
    placeholder="例: この映画は最高でした！"
)

# 分析の実行
if st.button("分析する") and text:
    with st.spinner("分析中..."):
        result = classifier(text)[0]

    label = result["label"]
    score = result["score"]

    # 結果の表示
    if label == "positive":
        st.success(f"ポジティブ（確信度: {score:.1%}）")
    elif label == "negative":
        st.error(f"ネガティブ（確信度: {score:.1%}）")
    else:
        st.info(f"ニュートラル（確信度: {score:.1%}）")